In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import sys
sys.path.insert(0,'/home/kat/Repos/SALSA/')

### Load in the cleaned HIV dataset.

In [2]:
tokens_salsa = '#%()+-0123456789<=>BCFHILNOPRSX[]cnos'
# tokens_hiv = 'Ub)gsP5u=8H[d]noW06%2GLlO+NFSTa9ZCeI-r4(1ihAVBRc3M7pt#'

df_hiv = pd.read_csv('data/model_ready/hiv/train/anchor_smiles.csv')
tokens_hiv = list(set(list(''.join(df_hiv.smiles.values))))
print(len(tokens_hiv), len(tokens_salsa))

34 37


### Load in models, run RF, save results.

In [3]:
from benchmark_utils import get_latents
from imblearn.ensemble import BalancedRandomForestClassifier as BalRF
from sklearn.model_selection import StratifiedKFold
from utilities.fp_utils import get_fps_in_parallel
import numpy as np

seedy = 666

def undo_BrCl_singles(smi):
    smi = smi.replace('R','Br')
    return smi.replace('L','Cl')
def do_BrCl_singles(smi):
    smi = smi.replace('Br','R')
    return smi.replace('Cl','L')   

# # # # # # # # # # # #
load_bs = 50
# # # # # # # # # # # #

tags = ['2022041804_04',  # salsa
        '2022041807_a03', # contrastive encoder 
        '2022041809_a04', # vanilla ae
        'morgan']         # morgan fingerprint

for tag in tags:
    
    if tag=='morgan':
        df_hiv = pd.read_csv('data/model_ready/hiv/train/anchor_smiles.csv')
        smis_hiv = [undo_BrCl_singles(sm) for sm in df_hiv.smiles]
#         fps = get_fps_in_parallel(smis_hiv,fp_type='morgan',counts=False,bits=1024,radius=2)
        fps = get_fps_in_parallel(smis_hiv,fp_type='morgan',counts=False,bits=2048,radius=5)
        X = np.stack(fps)
    else:
        X = get_latents(tag, 'train', load_bs) 
        
    y = pd.read_csv('data/model_ready/hiv/train/labels.csv')['y'].values

    ## for holding results ... 
    df_y = pd.DataFrame(y, columns=['y_test'])

    skf = StratifiedKFold(n_splits=5, random_state=seedy, shuffle=True)
    skf.get_n_splits(X, y)

    clf = BalRF(n_estimators=100, max_features="auto",sampling_strategy='auto',
                random_state=seedy,n_jobs=-1,oob_score=True)

    oobs = []
    for i,(train_idx, test_idx) in enumerate(skf.split(X,y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        clf.fit(X_train,y_train)
        print(f'OOB, split {i}: {clf.oob_score_}')

        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)
        df_y.loc[test_idx,'y_prob'] = y_prob[:,1]
        df_y.loc[test_idx,'y_pred'] = y_pred
        oobs.append(clf.oob_score_)

    df_y.to_csv(f'{tag}_hiv_benchmark.csv',index=False)
    print(f"Mean OOB: {sum(oobs)/5}")

Using 4 GPUs!
Loaded model weights from /home/kat/Repos/SALSA/results/models/2022041804_04/29.pt!


637it [01:56,  5.45it/s]                         


OOB, split 0: 0.6848121085594989
OOB, split 1: 0.6841858037578288
OOB, split 2: 0.6796450939457203
OOB, split 3: 0.6801299177536801
OOB, split 4: 0.6866359447004609
Mean OOB: 0.6830817737434378
Using 4 GPUs!
Loaded model weights from /home/kat/Repos/SALSA/results/models/2022041807_a03/29.pt!


637it [01:50,  5.79it/s]                         


OOB, split 0: 0.682098121085595
OOB, split 1: 0.6799582463465553
OOB, split 2: 0.6838726513569937
OOB, split 3: 0.6863638744826863
OOB, split 4: 0.6870548806032677
Mean OOB: 0.6838695547750197
Using 4 GPUs!
Loaded model weights from /home/kat/Repos/SALSA/results/models/2022041809_a04/29.pt!


637it [01:49,  5.80it/s]                         


OOB, split 0: 0.6670668058455115
OOB, split 1: 0.6646137787056368
OOB, split 2: 0.6586116910229645
OOB, split 3: 0.6599612342186599
OOB, split 4: 0.6690406367825723
Mean OOB: 0.663858829315069
OOB, split 0: 0.7070459290187892
OOB, split 1: 0.7018267223382046
OOB, split 2: 0.7034446764091858
OOB, split 3: 0.7024988213107025
OOB, split 4: 0.7120339338081274
Mean OOB: 0.7053700165770019


### Performance metrics.

In [7]:
from benchmark_utils import get_metrics
import pprint

versions = ['salsa','contrastive','vanilla AE','morgan']

for i,tag in enumerate(tags):
    df_y = pd.read_csv(f'{tag}_hiv_benchmark.csv')
    metrics = get_metrics(df_y.y_test, df_y.y_pred, df_y.y_prob)
    print(f'\n{versions[i]}')
    pprint.pprint(metrics)


salsa
{'AUROC': 0.7884108979903013,
 'Average Precision': 0.078060059114176,
 'BAcc': 0.7202026814806604,
 'Pr': 0.10082644628099173,
 'Sn': 0.6529884032114184,
 'Sp': 0.7874169597499023}

contrastive
{'AUROC': 0.7928375180704159,
 'Average Precision': 0.08221123383737375,
 'BAcc': 0.7274221910343779,
 'Pr': 0.1070854638422206,
 'Sn': 0.6538804638715433,
 'Sp': 0.8009639181972125}

vanilla AE
{'AUROC': 0.7693714103912459,
 'Average Precision': 0.07169670037190787,
 'BAcc': 0.706858054754653,
 'Pr': 0.0913884007029877,
 'Sn': 0.6494201605709188,
 'Sp': 0.7642959489383874}

morgan
{'AUROC': 0.8244287186362593,
 'Average Precision': 0.11419604197130323,
 'BAcc': 0.7667815498336783,
 'Pr': 0.15342072920934044,
 'Sn': 0.6681534344335415,
 'Sp': 0.8654096652338152}
